# C1.1 · Platform ingestion and supply-chain risks

**Function C — Red Teaming and Security Research with AI → Agentic Evaluation and Red Teaming**

Builds on **[C1.0 · Start here — the evolution of non-deterministic threat simulation](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**.

| | |
|---|---|
| Tools used | Sigstore, OSV |

## What this lesson is

**What it covers.** Auditing a model platform's ingress filters — dependency-squatting, malicious uploads to a hub, poisoned datasets — by assessing each component on whether it can change without notice.

**Why a security engineer needs it.** The supply chain is the surface an attacker reaches before the model, and popularity is not provenance: a hosted model or a re-pulled dataset can change under a name that never did, invalidating every control you tested against it.

## 1 · The hook

The first surface an attacker reaches is not the model — it is the hub the model was downloaded from. A squatted package name and a poisoned dataset both land before a single prompt is sent, and the platform accepted them because the name looked right.

> **At CyberTravels.** The components are CyberTravels' own — the model its advisor calls, the packages its coding agent installs, the templates indexed into its vector store — and the one with no pinned digest is the route in.

## 2 · The framework

```
   an attacker's first reach is the SUPPLY, not the model

   pip install helpf-ul-utils   squatted name   -> accepted
   model: org/whisper-turbo     never verified  -> accepted
   dataset re-pulled each train  changes freely  -> accepted

   assess on:  can this change without telling me?
     pinned library     no   -> low
     hosted model       yes  -> high
     re-pulled dataset  yes  -> high, and no digest = the finding
```

The first surface an attacker reaches is not the model — it is the **hub the
model was downloaded from**. Dependency-squatting a package name, uploading a
malicious model to a public registry, or shipping a poisoned dataset are all
supply-chain attacks that land before a single prompt is sent.

Red teaming this surface means auditing the ingress filters: what a platform
accepts, what it verifies, and what it takes on trust because the name looked
right. The research discipline is to assess a component on whether it can
**change without telling you** — a pinned library cannot, a hosted model can,
and a dataset re-pulled on every train can change under a name that never did.

## 3 · The procedure, as a skill

The skill scores each component CyberTravels pulls in on whether it can change without notice, and separates the pinned artefacts from the ones a registry can replace on a Tuesday.

### The skill — [`skills/research/agent-supply-chain-assessment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/agent-supply-chain-assessment/SKILL.md)

```yaml
name: agent-supply-chain-assessment
description: >-
  Score new packages and MCP connectors for typosquatting and ordinary
  supply-chain signals, then re-score them weighted by the authority the agent
  that loads them runs with. Use when an agent installs its own dependencies or
  connects to a server somebody added last week.
allowed-tools: Read, Grep, Glob
```

# The same package is a different risk inside an agent

Supply-chain assessment for agents differs in one term: the artefact runs with
the agent's authority. A connector that trips three ordinary signals is a
review; the same connector loaded by an agent holding production credentials is
a block. Weighting by authority is what turns the ordinary assessment into the
right answer.

## When to use this

When an agent can install packages, when an MCP server is added, and at any
review of what a coding agent is allowed to pull.

## Procedure

**1 — Establish the known-good set.** The packages this project actually uses.
Typosquat detection is a comparison against something; without the set it is a
spell-check.

**2 — Score edit distance against known-good names.** A distance of one or two
from a popular package, with a recent first-publication date, is the classic
shape. Report the package it imitates, not just the score.

**3 — Apply the ordinary signals.** Age, maintainer count, download history,
whether it was published after the agent asked for it, install scripts.

**4 — Re-score weighted by agent authority.** What credentials are in the
environment the artefact will execute in, and what the agent can reach. This is
the step that moves a review to a block and it needs no new information.

**5 — Report the two verdicts side by side.** Unweighted and authority-weighted.
The difference is the argument for gating what agents may install, and it is
easier to make with both numbers present.

## Example

**Input** — the fixture committed at the top of [`scripts/agent_supply_chain_assessment.py`](scripts/agent_supply_chain_assessment.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
requests==2.31.0        allow
requsts==2.31.0         block
      · unsigned — no attestation to source
      · published 3d ago — no soak time
      · only 12 downloads
      · distance 1 from popular package 'requests'
colourama==0.4.6        block
      · unsigned — no attestation to source
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "known_good": ["str"],
  "artefacts": [{"name": "str", "kind": "package|mcp", "distance": 0, "imitates": "str|null",
                 "signals": ["str"], "verdict": "allow|review|block"}],
  "authority": {"credentials_present": ["str"], "reachable": ["str"]},
  "weighted": [{"name": "str", "verdict": "allow|review|block", "moved": true}]
}
```

## Failure modes

- **Typosquat detection with no known-good set.** Everything is close to
  something.
- **Assessing the package and not the environment.** The authority is the term
  that differs.
- **Treating an MCP connector as configuration.** It is code that runs with the
  agent.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/agent-supply-chain-assessment/scripts/agent_supply_chain_assessment.py
SCRIPT = "skills/research/agent-supply-chain-assessment/scripts/agent_supply_chain_assessment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The pinned libraries score low-risk, the hosted model and the re-pulled dataset score high because neither can be pinned, and the one component with no provenance at all is the finding.

## Your turn

List every model and dataset your pipeline pulls at train or deploy time. The ones with no pinned digest are the ones an attacker can change without touching your code.

---

**Next → [C1.2 · Weaponizing the ingestion path](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*